# What the Fingers Hear — Correction Demo

Visualizes pitch, duration, and accidental corrections using the best trained M2BERT model for each task.

| Task | Model | Dev Accuracy |
|------|-------|-------------|
| Pitch | M2BERT joint (Nov-3) | 63.1% top-1 |
| Duration | M2BERT fine-tuned (Nov-3) | 88.2% top-1 |
| Accidental | M2BERT fine-tuned (v2) | 87.7% top-1 |

Sheet music rendered with [Verovio](https://www.verovio.org/).

In [ ]:
import verovio, json
from pathlib import Path
from IPython.display import HTML, display

DEMO_DIR = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/demo/inference_demo')
ML_DIR   = Path('/Users/zhaojinlan/Desktop/Lann Projects/What-the-fingers-hear/ml/demo')

def _render(xml_path, scale=35):
    tk = verovio.toolkit()
    tk.setOptions({
        'pageWidth': 1800,
        'adjustPageHeight': True,
        'footer': 'none',
        'header': 'none',
        'scale': scale,
    })
    tk.loadData(Path(xml_path).read_text())
    return tk.renderToSVG(1)

def show_comparison(title, in_path, out_path, pred_path, description='', scale=35):
    pred     = json.load(open(pred_path))
    in_svg   = _render(in_path, scale)
    out_svg  = _render(out_path, scale)

    cur = pred['current_value']
    prd = pred['prediction']
    top_k = pred['candidates'][:5]
    max_p = max(c['prob'] for c in top_k) or 1.0

    bars = ''.join(f'''
        <div style="display:flex;align-items:center;margin:3px 0">
          <span style="width:110px;text-align:right;font-size:12px;margin-right:8px;
                       font-family:monospace;color:{'#1a7' if c['label']==prd else '#555'};
                       font-weight:{'700' if c['label']==prd else '400'}">{c['label']}</span>
          <div style="background:{'#27ae60' if c['label']==prd else '#bdc3c7'};
                      height:12px;width:{int(c['prob']/max_p*140)}px;border-radius:2px"></div>
          <span style="margin-left:7px;font-size:11px;color:#888">{c['prob']:.1%}</span>
        </div>''' for c in top_k)

    html = f'''
<div style="font-family:sans-serif;margin:20px 0;padding:18px 20px;
             border:1px solid #d0d0d0;border-radius:10px;background:#fafafa">
  <h3 style="margin:0 0 6px;color:#1a1a2e;font-size:18px">{title}</h3>
  {'<p style="margin:0 0 14px;color:#666;font-size:13px">' + description + '</p>' if description else ''}
  <div style="display:flex;gap:18px;align-items:flex-start">

    <div style="flex:1;min-width:0">
      <div style="text-align:center;color:#c0392b;font-weight:600;font-size:13px;
                  margin-bottom:6px;letter-spacing:0.5px">INPUT (corrupted)</div>
      <div style="overflow-x:auto;border:1.5px solid #e74c3c;border-radius:6px;
                  background:white;padding:4px">{in_svg}</div>
    </div>

    <div style="flex:0 0 210px;background:white;padding:14px;border:1px solid #ddd;
                border-radius:8px;box-shadow:0 1px 4px rgba(0,0,0,.07)">
      <p style="font-weight:700;margin:0 0 10px;color:#1a1a2e;font-size:14px">Model prediction</p>
      <p style="margin:0 0 4px;font-size:12px;color:#555">
        Input: <code style="background:#ffeaea;padding:1px 4px;border-radius:3px">{cur}</code>
      </p>
      <p style="margin:0 0 12px;color:#27ae60;font-weight:700;font-size:15px">→ {prd}</p>
      {bars}
      <p style="margin:12px 0 0;font-size:11px;color:#aaa">{pred['context_notes']} context notes</p>
    </div>

    <div style="flex:1;min-width:0">
      <div style="text-align:center;color:#27ae60;font-weight:600;font-size:13px;
                  margin-bottom:6px;letter-spacing:0.5px">OUTPUT (corrected)</div>
      <div style="overflow-x:auto;border:1.5px solid #27ae60;border-radius:6px;
                  background:white;padding:4px">{out_svg}</div>
    </div>

  </div>
</div>'''
    display(HTML(html))

print('Setup complete — verovio', verovio.toolkit().getVersion())

---
## 1. Pitch Correction
The model identifies a wrong note in context and predicts the correct pitch from 52 possible classes.

In [ ]:
P12  = ML_DIR / 'unified_joint' / 'pitch_12_samples_5023603_p4-s3'
P72  = ML_DIR / 'unified_joint' / 'pitch_72_samples_5986599_p4-s3'
P327 = ML_DIR / 'unified_joint' / 'pitch_327_samples_6474469_p1-s1'

show_comparison(
    'Pitch Correction 1 — C4 → E4',
    P12 / 'corrupted.musicxml',
    P12 / 'model_corrected.musicxml',
    P12 / 'prediction.json',
    description='A note written as C4 is corrected to E4. '
                'The model selects from 52 pitch classes with 57 surrounding context notes.',
)

show_comparison(
    'Pitch Correction 2 — G1 → B2',
    P72 / 'corrupted.musicxml',
    P72 / 'model_corrected.musicxml',
    P72 / 'prediction.json',
    description='A note displaced by over two octaves (G1 instead of B2) is recovered '
                'from 61 context notes.',
)

show_comparison(
    'Pitch Correction 3 — D5 → A1',
    P327 / 'corrupted.musicxml',
    P327 / 'model_corrected.musicxml',
    P327 / 'prediction.json',
    description='A bass note displaced four octaves (D5 instead of A1) is correctly '
                'restored from 80 context notes.',
)

---
## 2. Duration Correction
The model identifies a note with an incorrect rhythmic value and restores the correct duration from 28 duration classes.

In [ ]:
D1 = ML_DIR / 'unified_joint' / 'dur_new_samples_6425381_p4-s3'
D2 = ML_DIR / 'unified_joint' / 'dur_new_samples_6259566_p3-s4'
D3 = ML_DIR / 'unified_joint' / 'dur_new_samples_5001716_p4-s3'

show_comparison(
    'Duration Correction 1 — 4 QL → 1/4 QL',
    D1 / 'corrupted.musicxml',
    D1 / 'model_corrected.musicxml',
    D1 / 'prediction.json',
    description='A note erroneously written as a whole note (4 QL) is corrected to a '
                'sixteenth note (1/4 QL).',
)

show_comparison(
    'Duration Correction 2 — 4 QL → 1/4 QL',
    D2 / 'corrupted.musicxml',
    D2 / 'model_corrected.musicxml',
    D2 / 'prediction.json',
    description='A note erroneously written as a whole note (4 QL) is corrected to a '
                'sixteenth note (1/4 QL).',
)

show_comparison(
    'Duration Correction 3 — 4 QL → 1/8 QL',
    D3 / 'corrupted.musicxml',
    D3 / 'model_corrected.musicxml',
    D3 / 'prediction.json',
    description='A note erroneously written as a whole note (4 QL) is corrected to a '
                'thirty-second note (1/8 QL).',
)

---
## 3. Accidental Correction
The model identifies an incorrect accidental (sharp / flat / natural) and predicts the correct one from 5 classes: ♭♭, ♭, ♮, ♯, ×.

In [ ]:
show_comparison(
    'Accidental Correction — ♭ → ♯',
    DEMO_DIR / 'acc_test_+1.musicxml',
    DEMO_DIR / 'acc_output_+1' / 'model_corrected.musicxml',
    DEMO_DIR / 'acc_output_+1' / 'prediction.json',
    description='A note erroneously written with a flat (♭) is corrected to a sharp (♯).',
)

show_comparison(
    'Accidental Correction — ♮ → ♭',
    DEMO_DIR / 'acc_test_-1.musicxml',
    DEMO_DIR / 'acc_output_-1' / 'model_corrected.musicxml',
    DEMO_DIR / 'acc_output_-1' / 'prediction.json',
    description='A note missing its flat — written as natural (♮) — is correctly identified '
                'and restored to flat (♭).',
)

show_comparison(
    'Accidental Correction — ♭ → ♮',
    DEMO_DIR / 'acc_test_+0.musicxml',
    DEMO_DIR / 'acc_output_+0' / 'model_corrected.musicxml',
    DEMO_DIR / 'acc_output_+0' / 'prediction.json',
    description='A note incorrectly marked with a flat (♭) is corrected to natural (♮).',
)